In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

In [3]:
df = pd.read_excel('./data_adults_glmer_dummies.xlsx')

In [4]:
df.columns

Index(['Subject', 'Item', 'Suffix', 'Response_apu', 'Response_pu'], dtype='str')

In [5]:
df.Suffix.value_counts()

Suffix
iF     945
oN     945
osM    945
iN     945
maN    945
isM    945
aF     945
Name: count, dtype: int64

In [6]:
df_suffix = df[df['Suffix'] =='aF']

In [7]:
df_suffix.head()

,Subject,Item,Suffix,Response_apu,Response_pu
12,116_A,Η ΚΗΒΛΙΚΑ,aF,0,1
15,116_A,Η ΕΠΟΥΦΑ,aF,1,0
16,116_A,Η ΟΥΛΙΒΑ,aF,0,1
17,116_A,Η ΡΙΜΕΘΑ,aF,1,0
21,116_A,Η ΚΛΟΜΕΖΑ,aF,1,0


In [8]:
np.sum(df_suffix.Response_apu)/len(df_suffix.Response_apu)

np.float64(0.2698412698412698)

In [9]:
model_apu = sm.BinomialBayesMixedGLM.from_formula(
    "Response_apu ~ 1",
    {
        "participant": "0 + C(Subject)",
        "item": "0 + C(Item)"
    },
    data=df_suffix
)


In [10]:
model_pu = sm.BinomialBayesMixedGLM.from_formula(
    "Response_pu ~ 1",
    {
        "participant": "0 + C(Subject)",
        "item": "0 + C(Item)"
    },
    data=df_suffix
)

In [11]:
def intercept_probability(result):
    b0 = result.fe_mean[0]
    b0_sd = result.fe_sd[0]

    def invlogit(x):
        return 1 / (1 + np.exp(-x))

    p = invlogit(b0)
    lower = invlogit(b0 - 1.96 * b0_sd)
    upper = invlogit(b0 + 1.96 * b0_sd)

    return p, lower, upper

In [12]:
result_apu = model_apu.fit_vb()
p, lower, upper = intercept_probability(result_apu)
print(f"Mean apu intercept probability: {p:.4f}")
print(f"95% credible interval: [{lower:.4f}, {upper:.4f}]")


Mean apu intercept probability: 0.2256
95% credible interval: [0.1990, 0.2545]


In [13]:
result_pu = model_pu.fit_vb()
p, lower, upper = intercept_probability(result_pu)
print(f"Mean pu intercept probability: {p:.4f}")
print(f"95% credible interval: [{lower:.4f}, {upper:.4f}]")

Mean pu intercept probability: 0.7593
95% credible interval: [0.7293, 0.7870]


In [19]:
sorted(df['Suffix'].values.unique())

['aF', 'iF', 'iN', 'isM', 'maN', 'oN', 'osM']

In [20]:
results = []
for suffix in sorted(df['Suffix'].values.unique()):
    df_suffix = df[df['Suffix'] == suffix]
    print(f"Analyzing suffix: {suffix}")
    
    model_apu = sm.BinomialBayesMixedGLM.from_formula(
        "Response_apu ~ 1",
        {
            "participant": "0 + C(Subject)",
            "item": "0 + C(Item)"
        },
        data=df_suffix
    )
    
    result_apu = model_apu.fit_vb()
    p, lower, upper = intercept_probability(result_apu)
    print(f"Mean apu intercept probability: {p:.4f}")
    print(f"95% credible interval: [{lower:.4f}, {upper:.4f}]")
    results.append({'Suffix': suffix, 'Property': 'APU', 'Mean intercept': p, 'CI low': lower, 'CI High': upper})
    
    model_pu = sm.BinomialBayesMixedGLM.from_formula(
        "Response_pu ~ 1",
        {
            "participant": "0 + C(Subject)",
            "item": "0 + C(Item)"
        },
        data=df_suffix
    )
    
    result_pu = model_pu.fit_vb()
    p, lower, upper = intercept_probability(result_pu)
    print(f"Mean pu intercept probability: {p:.4f}")
    print(f"95% credible interval: [{lower:.4f}, {upper:.4f}]")
    results.append({'Suffix': suffix, 'Property': 'PU', 'Mean intercept': p, 'CI low': lower, 'CI High': upper})
    
    print("-" * 40)

df_results = pd.DataFrame(results)
df_results

Analyzing suffix: aF
Mean apu intercept probability: 0.2256
95% credible interval: [0.1990, 0.2545]
Mean pu intercept probability: 0.7593
95% credible interval: [0.7293, 0.7870]
----------------------------------------
Analyzing suffix: iF
Mean apu intercept probability: 0.1078
95% credible interval: [0.0905, 0.1279]
Mean pu intercept probability: 0.8055
95% credible interval: [0.7786, 0.8298]
----------------------------------------
Analyzing suffix: iN
Mean apu intercept probability: 0.0355
95% credible interval: [0.0276, 0.0454]
Mean pu intercept probability: 0.9324
95% credible interval: [0.9176, 0.9446]
----------------------------------------
Analyzing suffix: isM
Mean apu intercept probability: 0.0223
95% credible interval: [0.0163, 0.0305]
Mean pu intercept probability: 0.8317
95% credible interval: [0.8058, 0.8547]
----------------------------------------
Analyzing suffix: maN
Mean apu intercept probability: 0.9485
95% credible interval: [0.9356, 0.9588]
Mean pu intercept prob

,Suffix,Property,Mean intercept,CI low,CI High
0,aF,APU,0.225551,0.198980,0.254542
1,aF,PU,0.759305,0.729282,0.786970
2,iF,APU,0.107787,0.090508,0.127900
3,iF,PU,0.805470,0.778608,0.829785
4,iN,APU,0.035484,0.027648,0.045437
5,iN,PU,0.932358,0.917580,0.944647
6,isM,APU,0.022312,0.016276,0.030518
7,isM,PU,0.831691,0.805827,0.854730
8,maN,APU,0.948452,0.935611,0.958844
9,maN,PU,0.047155,0.037401,0.059295


In [21]:
df_results.to_excel('./GLM_Adults_intercept_probabilities.xlsx', index=False)